In [ ]:
%%configure
{
  "defaultLakehouse": {
    "name": { "parameterName": "LakehouseName", "defaultValue": "AzureCostAnalyzer_LH" },
    "id": { "parameterName": "LakehouseId", "defaultValue": "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx" },
    "workspaceId": { "parameterName": "WorkspaceId", "defaultValue": "yyyyyyyy-yyyy-yyyy-yyyy-yyyyyyyyyyyy" }
  }
}

# 08 · Gold · Cost by Resource

Builds **`gold_cost_by_resource`** — cost at **individual-resource grain** for the **Explorer /
Resource Detail / Governance** views.

**Tags no longer live here.** The dynamic tag model is **`gold_chargeback_by_tag`** (WIDE) +
**`dim_tag_key`**, built in notebook **05**. This table keeps only **`TagCount`** (`0` = fully
untagged) so the Governance measures/KPIs work with **no tag columns**, and stays lean for the
non-tag views.

**Output grain**: `YearMonth × SubAccountName × ResourceGroupName × ResourceId × ResourceName ×
ResourceType × ServiceCategory × ServiceName × RegionName`

**Metrics**: `EffectiveCost`, `BilledCost`, `ListCost`, `SavingsAmount` (hybrid baseline:
`ContractedCost` for committed usage, else `ListCost`, minus `EffectiveCost`), `TagCount`.

**Dependencies**: `focus_partitioned` (silver) with `Tags` JSON, `ResourceId`, `ResourceName`,
`PricingCategory`, `ContractedCost`.

> Standardized to the validated ACA model: the old top-6 tag columns and the `gold_tag_keys` helper
> are **removed** (replaced by the WIDE `gold_chargeback_by_tag` + `dim_tag_key` in notebook 05).
> Only `TagCount` remains, so `Untagged Cost/%`, `Untagged Resource Count`, etc. still work.


In [ ]:
# Build gold_cost_by_resource — resource-grain cost for Explorer / Resource Detail / Governance.
# TAGS NO LONGER LIVE HERE — the dynamic tag model is `gold_chargeback_by_tag` (WIDE) + `dim_tag_key`
# (built in notebook 05). We keep only `TagCount` (0 = fully untagged, from the ORIGINAL Tags JSON)
# so the Governance measures/KPIs work with NO tag columns, and this table stays lean for the
# non-tag views. Hybrid savings baseline matches notebook 03 (ContractedCost for committed usage,
# else ListCost, minus EffectiveCost).
# mssparkutils / notebookutils are available globally in the Fabric runtime (no import needed)
from datetime import date
from dateutil.relativedelta import relativedelta
from pyspark.sql import functions as F

# Lower-casing keys could collapse case-variants; make the last value win instead of raising
# "Duplicate map key" when parsing the Tags JSON.
spark.conf.set("spark.sql.mapKeyDedupPolicy", "LAST_WIN")

# Partition pruning: read last 12 months (align with the other gold notebooks)
lookback_date = date.today() - relativedelta(months=12)
year_filter, month_filter = lookback_date.year, lookback_date.month

def _clean(expr, default):
    c = F.trim(expr)
    return F.when(c.isNull() | (c == ""), F.lit(default)).otherwise(c)

usage = (spark.table("focus_partitioned")
    .where((F.col("ChargeCategory") == "Usage")
           & ((F.col("Year") > year_filter) | ((F.col("Year") == year_filter) & (F.col("Month") >= month_filter))))
    .withColumn("YearMonth", F.date_format("ChargePeriodStart", "yyyy-MM")))

# TagCount = size of the FULL tag map (all real tags on the resource), aggregated as MAX per
# resource so a resource is "untagged" only when it never had any tag.
_rt = usage.withColumn("tagmap", F.from_json("Tags", "map<string,string>"))
_r2 = (_rt
    .withColumn("_TagMapSize", F.when(F.col("tagmap").isNull(), F.lit(0)).otherwise(F.size("tagmap")))
    .withColumn("_BaselineRow", F.when(F.col("PricingCategory") == "Committed", F.col("ContractedCost")).otherwise(F.col("ListCost")))
    .withColumn("_ResourceType",
        _clean(F.regexp_extract(F.lower(F.col("ResourceId")), r"providers/([^/]+/[^/]+)", 1), "(unknown)")))

_group = ["YearMonth", "SubAccountName",
    _clean(F.col("x_ResourceGroupName"), "(no resource group)").alias("ResourceGroupName"),
    _clean(F.col("ResourceId"), "(no resource)").alias("ResourceId"),
    _clean(F.col("ResourceName"), "(no resource)").alias("ResourceName"),
    F.col("_ResourceType").alias("ResourceType"),
    "ServiceCategory", "ServiceName", "RegionName"]
gold_cost_by_resource = (_r2.groupBy(*_group)
    .agg(F.sum("EffectiveCost").alias("EffectiveCost"), F.sum("BilledCost").alias("BilledCost"),
         F.sum("ListCost").alias("ListCost"),
         (F.sum("_BaselineRow") - F.sum("EffectiveCost")).alias("SavingsAmount"),
         F.max("_TagMapSize").alias("TagCount")))
gold_cost_by_resource.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_cost_by_resource")
print(f"✓ Created gold_cost_by_resource with {gold_cost_by_resource.count():,} rows (resource grain, tags removed, TagCount kept)")

# Retire the old gold_tag_keys helper (replaced by dim_tag_key from notebook 05).
if spark.catalog.tableExists("gold_tag_keys"):
    spark.sql("DROP TABLE IF EXISTS gold_tag_keys")
    print("Dropped retired gold_tag_keys (replaced by dim_tag_key in notebook 05).")


## Validate output

Confirm the schema, row count, cost total, and the **governance** signal via `TagCount`
(`0` = fully untagged). The tag universe now lives in **`dim_tag_key`** (notebook 05).


In [ ]:
from pyspark.sql import functions as F

g = spark.table("gold_cost_by_resource")

# Schema + headline stats. Cost columns are decimal(38,18) → Spark returns them as Python
# `decimal.Decimal`; `UntaggedCost` mixes a double literal so it comes back as `float`. Cast
# both to float before dividing/formatting (float / Decimal raises TypeError).
g.printSchema()
stats = g.select(
    F.count("*").alias("Rows"),
    F.countDistinct("ResourceId").alias("Resources"),
    F.countDistinct("ServiceName").alias("Services"),
    F.sum("EffectiveCost").alias("TotalEffectiveCost"),
    F.sum(F.when(F.col("TagCount") == 0, F.col("EffectiveCost")).otherwise(F.lit(0.0))).alias("UntaggedCost"),
    F.countDistinct(F.when(F.col("TagCount") == 0, F.col("ResourceId"))).alias("UntaggedResources"),
).first()
_tot = float(stats.TotalEffectiveCost or 0)
_unt = float(stats.UntaggedCost or 0)
_pct = (_unt / _tot * 100) if _tot else 0
print("=" * 70)
print(f"Rows                 : {stats.Rows:,}")
print(f"Distinct resources   : {stats.Resources:,}")
print(f"Distinct services    : {stats.Services:,}")
print(f"Total Effective Cost : ${_tot:,.2f}")
print(f"Untagged (TagCount=0): ${_unt:,.2f}  ({_pct:.1f}%)  ·  {stats.UntaggedResources:,} resources")
print("=" * 70)

# The dynamic tag model now lives in gold_chargeback_by_tag (WIDE) + dim_tag_key (notebook 05).
print("\nTag universe (dim_tag_key — built in notebook 05):")
if spark.catalog.tableExists("dim_tag_key"):
    spark.table("dim_tag_key").orderBy("Rank").show(50, truncate=False)
else:
    print("  dim_tag_key not found — run notebook 05 (05_Gold_ByTag) first.")

# Generic drill: top resources by cost.
print("\nTop 15 resources by Effective Cost:")
(g.groupBy("ResourceName", "ResourceType", "ServiceName")
  .agg(F.round(F.sum("EffectiveCost"), 2).alias("EffectiveCost"))
  .orderBy(F.desc("EffectiveCost"))
  .show(15, truncate=False))
